In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


REPO_ROOT = Path("..").resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.io import load_dataset, load_fine


DATA_DIR = REPO_ROOT / "DATA"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

fine = load_fine(
    DATA_DIR,
    "fine.csv",
)

print(f"Repository root : {REPO_ROOT}")
print(f"Data directory  : {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Load the standardized fine-level dataset used for PCA and define the PCA and
# K-means parameters

FIGURE_DIR = REPO_ROOT / "Fig_1"

pca_data = load_dataset(
    FIGURE_DIR,
    "PCA_fine_level_df_standardized.csv",
    index_col=0,
)

n_components = 3

pca = PCA(
    n_components=n_components,
    random_state=42,
)

pca_scores = pca.fit_transform(pca_data)

kmeans = KMeans(
    n_clusters=4,
    n_init=50,
    random_state=42,
)

cluster_labels = kmeans.fit_predict(pca_scores)

pc_names = [f"PC{i}" for i in range(1, n_components + 1)]

pca_scores_df = pd.DataFrame(
    pca_scores,
    index=pca_data.index,
    columns=pc_names,
)

pca_scores_df["Cluster"] = cluster_labels.astype(str)
pca_scores_df["Label"] = pca_scores_df.index

pca_loadings = pd.DataFrame(
    np.abs(pca.components_.T),
    index=pca_data.columns,
    columns=pc_names,
)

pca_loadings.index = fine.index
pca_loadings = pca_loadings.droplevel(["mid", "fine"])

In [ ]:
# Identify the top-loading regions for each principal component and summarize
# their frequency and proportion by anatomical area

top_n = 20

top_loadings = pd.concat(
    [
        pca_loadings.nlargest(top_n, pc).assign(PC=pc)
        for pc in pc_names
    ]
)

top_loadings["Area"] = top_loadings.index.str.split("_").str[0]

loading_frequency = (
    top_loadings
    .groupby(["PC", "Area"])
    .size()
    .reset_index(name="Count")
)

loading_frequency["Proportion"] = (
    loading_frequency["Count"] / top_n
)

loading_frequency["Area_PC"] = (
    loading_frequency["Area"]
    + " ("
    + loading_frequency["PC"]
    + ")"
)

loading_frequency = loading_frequency.sort_values(
    ["PC", "Proportion"],
    ascending=[True, False],
)

In [ ]:
# Plot the relative frequency of top-loading brain regions across PCs

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 24,
})

fig_height = max(6, len(loading_frequency) * 0.4)

fig, ax = plt.subplots(figsize=(10, fig_height))

sns.barplot(
    data=loading_frequency,
    x="Proportion",
    y="Area_PC",
    hue="PC",
    dodge=False,
    palette="Set2",
    ax=ax,
)

ax.set(
    xlabel="Relative Frequency",
    ylabel="Brain Region",
    xlim=(0, 1),
)

ax.set_yticklabels(loading_frequency["Area"])

plt.xticks(fontsize=20)
plt.yticks(fontsize=20)

ax.legend(loc="upper right")

plt.tight_layout()

output_file = OUTPUT_DIR / "PCs_combined.svg"

fig.savefig(output_file, format="svg", bbox_inches="tight")

print(f"Figure ready: {output_file}")

plt.show()

In [ ]:
# Visualize the four clusters in the three-dimensional PCA space

from utils.plot import plot_covariance_ellipsoid

plot_data = pca_scores_df.copy()
plot_data["Cluster"] = plot_data["Cluster"].astype(int)

n_clusters = plot_data["Cluster"].nunique()
cluster_colors = plt.cm.viridis(np.linspace(0, 1, n_clusters))

show_labels = False

fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection="3d")

for cluster_id, color in enumerate(cluster_colors):

    cluster = plot_data[plot_data["Cluster"] == cluster_id]

    ax.scatter(
        cluster["PC1"],
        cluster["PC2"],
        cluster["PC3"],
        color=color,
        label=f"Cluster {cluster_id}",
        s=60,
        alpha=0.95,
        edgecolors="black",
    )

    centroid = cluster[["PC1", "PC2", "PC3"]].mean()

    ax.scatter(
        centroid["PC1"],
        centroid["PC2"],
        centroid["PC3"],
        marker="X",
        s=120,
        color="black",
        edgecolors="white",
        linewidths=1.5,
    )

    if show_labels:
        for _, row in cluster.iterrows():
            ax.text(
                row["PC1"],
                row["PC2"],
                row["PC3"],
                row["Label"],
                fontsize=7,
                ha="left",
                va="bottom",
            )

    if len(cluster) >= 4:
        plot_covariance_ellipsoid(
            ax=ax,
            center=centroid.to_numpy(),
            covariance=np.cov(
                cluster[["PC1", "PC2", "PC3"]].to_numpy(),
                rowvar=False,
            ),
            color=color,
        )

ax.set(
    xlabel="PC1",
    ylabel="PC2",
    zlabel="PC3",
)

ax.xaxis.labelpad = 25
ax.yaxis.labelpad = 25
ax.zaxis.labelpad = 25

for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.pane.set_facecolor((1, 1, 1, 1))

ax.grid(True)
ax.set_box_aspect(None, zoom=0.78)
ax.view_init(elev=20, azim=140)

plt.tight_layout()

output_file = OUTPUT_DIR / "pca_3d_clusters.svg"

fig.savefig(output_file, format="svg", bbox_inches="tight", pad_inches=0.5)

print(f"Figure ready: {output_file}")

plt.show()

In [ ]:
# Plot the coarse-level c-Fos dataset as a heatmap

coarse_data = (
    load_dataset(
        DATA_DIR,
        "coarse.csv",
        index_col=0,
    )
)

plt.rcParams.update(
    {
        "font.family": "Arial",
        "font.size": 25,
    }
)

fig, ax = plt.subplots(figsize=(12, 8))

heatmap = sns.heatmap(
    coarse_data,
    xticklabels=False,
    cmap="magma",
    cbar_kws={
        "label": r"c-Fos$^+$/mm$^2$",
    },
    ax=ax,
)


group_positions = {}

for column_index, region_name in enumerate(coarse_data.columns):
    group_name = "".join(
        character for character in region_name
        if not character.isdigit()
    )

    group_positions.setdefault(group_name, []).append(column_index)


for group_index, indices in enumerate(group_positions.values()):
    if group_index > 0:
        ax.axvline(
            x=indices[0],
            color="white",
            linewidth=2,
        )


label_y_position = heatmap.get_ylim()[0] + 0.45

for group_name, indices in group_positions.items():
    group_center = (indices[0] + indices[-1]) / 2

    ax.text(
        group_center + 0.5,
        label_y_position,
        group_name,
        ha="center",
        va="center",
        fontsize=23,
    )


ax.set_ylabel("")

ax.set_yticklabels(
    ax.get_yticklabels(),
    fontsize=22,
    rotation=45,
)

ax.tick_params(
    axis="y",
    left=True,
)


colorbar = heatmap.collections[0].colorbar

colorbar.ax.tick_params(
    labelsize=23,
    width=1.5,
)

colorbar.set_label(
    r"c-Fos$^+$/mm$^2$",
    fontsize=23,
    rotation=90,
    labelpad=20,
)


plt.tight_layout()

output_file = OUTPUT_DIR / "coarse_heatmap.svg"

fig.savefig(output_file, format="svg", bbox_inches="tight")

print(f"Figure ready: {output_file}")

plt.show()